In [1]:
import os, pickle, numpy as np, pandas as pd
from scipy import sparse
from implicit.als import AlternatingLeastSquares

ALS_DIR   = "C:/OPS/hybrid_filtering/files/ALS"
MODEL_PKL = os.path.join(ALS_DIR, "als_full.pkl")
os.makedirs(ALS_DIR, exist_ok=True)

# ─────────────────────────────────────────────
# A. 데이터 준비
# ─────────────────────────────────────────────
df = pd.read_csv("processed/processed_data_with_actr_disp.csv")
df.dropna(subset=["smry"], inplace=True)

# view_ratio → weight
bins   = [0, 0.25, 0.5, 0.75, 1.0]
labels = [1., 2., 3., 4.]
df["weight"] = pd.cut(df["view_ratio"], bins=bins,
                      labels=labels, include_lowest=True).astype(float)

In [3]:
mart = pd.read_csv('C:/OPS/hybrid_rec(test)/processed/vod_mart_processed.csv')
poster = pd.read_csv("C:/OPS/df_mart_posters.csv")

# 1) poster에서 super_asset_id별로 대표 포스터 한 건만 남깁니다.
poster_uniq = poster.drop_duplicates('super_asset_id')[['super_asset_id', 'poster_url']]

# 2) Series로 바꿔서 map ─ 메모리를 거의 쓰지 않습니다.
mart['poster_url'] = mart['super_asset_id'].map(
    poster_uniq.set_index('super_asset_id')['poster_url']
)

# 1) mart에서 asset_id별로 poster_url이 여러 개라면 중복 제거
poster_map = (
    mart.drop_duplicates('asset_id')[['asset_id', 'poster_url']]
         .set_index('asset_id')['poster_url']
)

# 2) map으로 붙이기 ─ df와 크기가 같은 1차원 배열만 생성하므로 메모리 효율 ↑
df['poster_url'] = df['asset_id'].map(poster_map)

C:\Users\user\AppData\Local\Temp\ipykernel_18632\2250822710.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  mart = pd.read_csv('C:/OPS/hybrid_rec(test)/processed/vod_mart_processed.csv')


In [7]:
# Check if 'orgnl_cntry' column already exists in df
if 'orgnl_cntry' not in df.columns:
    # Create a mapping from asset_id to orgnl_cntry
    country_map = mart[['asset_id', 'orgnl_cntry']].drop_duplicates().set_index('asset_id')['orgnl_cntry']
    
    # Map the country to df using the asset_id
    df['orgnl_cntry'] = df['asset_id'].map(country_map)
    
    print(f"✅ Added 'orgnl_cntry' column to df. Null values: {df['orgnl_cntry'].isna().sum()} out of {len(df)}")
else:
    print("⚠️ 'orgnl_cntry' column already exists in df.")

✅ Added 'orgnl_cntry' column to df. Null values: 0 out of 17163245


In [4]:
# ─────────────────────────────────────────────
# B. ALS 학습 & 저장 (없으면 학습, 있으면 건너뜀)
# ─────────────────────────────────────────────
if not os.path.exists(MODEL_PKL):
    print("📌 모델이 없으므로 새로 학습합니다…")

    # 1) 매핑
    user_map = {u: i for i, u in enumerate(df["user_index"].unique())}
    item_map = {a: i for i, a in enumerate(df["asset_id"].unique())}

    df["u_idx"] = df["user_index"].map(user_map)
    df["i_idx"] = df["asset_id"].map(item_map)

    # 2) 희소 행렬
    item_user = sparse.coo_matrix(
        (df["weight"].astype(np.float32),
         (df["i_idx"], df["u_idx"])),
        shape=(len(item_map), len(user_map)),
        dtype=np.float32
    ).tocsr()
    
    user_items = item_user.T

    # 3) 모델 학습
    model = AlternatingLeastSquares(
        factors=100, regularization=0.1,
        iterations=50, calculate_training_loss=True, use_gpu=False
    )
    model.fit(item_user, show_progress=True)

    # 4) 모델·매핑 저장
    payload = {
        "model": model,
        "user_map": user_map,
        "item_map": item_map,
        "bucket_bins": bins,
    }
    with open(MODEL_PKL, "wb") as f:
        pickle.dump(payload, f)
    print(f"✅ 모델 저장 완료 → {MODEL_PKL}")

else:
    print("✅ 기존 모델이 이미 존재하여 학습 단계 생략")

# ─────────────────────────────────────────────
# C. 모델·매핑 불러오기
# ─────────────────────────────────────────────
with open(MODEL_PKL, "rb") as f:
    payload = pickle.load(f)

als_model = payload["model"]
user_map  = payload["user_map"]
item_map  = payload["item_map"]
item_map_inv = {i: a for a, i in item_map.items()}

# (필요 시) 데이터프레임을 매핑과 동기화
df = df[df["user_index"].isin(user_map) & df["asset_id"].isin(item_map)].copy()
df["u_idx"] = df["user_index"].map(user_map)
df["i_idx"] = df["asset_id"].map(item_map)

# item_user / user_items 재생성
item_user = sparse.coo_matrix(
    (df["weight"].astype(np.float32),
     (df["i_idx"], df["u_idx"])),
    shape=(len(item_map), len(user_map)),
    dtype=np.float32
).tocsr()
user_items = item_user.T.tocsr()     # (users × items) 형태

✅ 기존 모델이 이미 존재하여 학습 단계 생략


In [5]:
# ================================================================
# E. 콘텐츠 데이터 전처리 · 형태소 분석
# ================================================================
import os, time, json, pickle
import pandas as pd
from mecab import MeCab

TFIDF_DIR = "C:/OPS/hybrid_filtering/files/TF-IDF"
os.makedirs(TFIDF_DIR, exist_ok=True)

# 1) 콘텐츠 원본 추출 & 중복 제거
content_df = (
    df[["asset_id", "super_asset_nm", "smry", "genre", "actr_disp"]]
    .drop_duplicates()
)

# 2) 그룹화(시리즈·시즌 → 하나의 group_id)
def group_content(df_):
    grp = (
        df_.groupby(["super_asset_nm", "smry"])["asset_id"]
        .apply(list).reset_index()
    )
    grp["group_id"] = range(len(grp))
    id2grp = {
        aid: gid
        for _, (ids, gid) in grp[["asset_id", "group_id"]].iterrows()
        for aid in ids
    }
    df_ = df_.copy()
    df_["group_id"] = df_["asset_id"].map(id2grp)
    return df_, grp

content_df, grp_meta = group_content(content_df)
rep_df   = content_df.drop_duplicates("group_id")
rep_map  = rep_df.set_index("group_id").to_dict("index")

# 3) 형태소 분석 (MeCab)
mecab = MeCab()

# 불용어 목록 확장
stopwords = {
    # 일반 조사/대명사
    "이", "그", "저", "등", "를", "을", "에", "에서", "와", "과", "의", "로", "으로", "이", "가", "은", "는",
    # 자주 나오는 동사/형용사
    "하다", "있다", "되다", "없다", "않다", "이다", "한다", "된다", "아니다", "보다",
    # 접속사
    "그리고", "그러나", "또한", "하지만", "또", "및", "이어", "혹은", "또는",
    # 미사/접미사
    "들", "씩", "적", "형", "같은", "같이", "처럼", "대한",
    # 숫자 관련
    "일", "이", "삼", "사", "오", "십", "백", "천", "만", "억", "첫", "두", "세", "네",
    # 시간 관련
    "년", "월", "일", "시간", "오전", "오후", "분", "초",
    # 방향 관련
    "위", "아래", "좌", "우", "상", "하", "전", "후", "중", "내", "외",
    # 빈도 높은 단어들
    "것", "때", "데", "곳", "수", "내", "말", "일", "그것", "무엇", "어떤", "이런", "그런", "저런"
}

# 형태소 분석 함수 (명사, 형용사, 동사만 추출)
def analyze_text(text:str) -> str:
    """입력 텍스트에서 명사(NN), 형용사(VA), 동사(VV)만 추출하여 공백으로 구분된 문자열 반환"""
    if pd.isna(text) or text == "":
        return ""
    
    # mecab으로 형태소 분석 후 원하는 품사만 필터링
    words = [
        w for w, t in mecab.pos(text)
        if t.startswith(("NN", "VA", "VV")) and len(w) > 1 and w not in stopwords
    ]
    return " ".join(words)

# 문서 생성 함수 (smry와 genre 형태소 분석 후 actr_disp와 결합)
def build_document(row):
    """행에서 smry, genre, actr_disp를 추출하여 형태소 분석 후 결합한 문서 생성"""
    # 장르는 가중치를 두기 위해 여러 번 반복
    genre_text = ""
    if pd.notna(row["genre"]):
        # 장르는 형태소 분석 없이 그대로 사용 (장르는 보통 단일어)
        genre_text = row["genre"] + " " + row["genre"] + " " + row["genre"]
    
    # 줄거리 형태소 분석
    smry_analyzed = analyze_text(row["smry"] or "")
    
    # 배우 정보는 그대로 사용
    actor_text = row["actr_disp"] if pd.notna(row["actr_disp"]) else ""
    
    # 분석된 텍스트 결합
    return " ".join(filter(None, [smry_analyzed, genre_text, actor_text]))

docs_path = os.path.join(TFIDF_DIR, "analyzed_texts.pkl")
if not os.path.exists(docs_path):
    analyzed_texts = {
        gid: build_document(pd.Series(info))
        for gid, info in rep_map.items()
    }
    with open(docs_path, "wb") as f:
        pickle.dump(analyzed_texts, f)
    print(f"✅ 형태소 분석 완료 → {docs_path}")
else:
    with open(docs_path, "rb") as f:
        analyzed_texts = pickle.load(f)
    print(f"🔄 형태소 분석 스킵 (이미 {len(analyzed_texts)}개 저장)")

# ================================================================
# F. TF-IDF 벡터화 및 저장
# ================================================================
vec_path = os.path.join(TFIDF_DIR, "tfidf_vectorizer.pkl")
gv_path  = os.path.join(TFIDF_DIR, "group_vectors.pkl")

if not os.path.exists(vec_path):
    from sklearn.feature_extraction.text import TfidfVectorizer
    group_ids  = list(analyzed_texts.keys())
    documents  = [analyzed_texts[gid] for gid in group_ids]

    # TF-IDF 벡터화 (파라미터 조정)
    tfidf = TfidfVectorizer(
        min_df=3, max_df=0.9,  # 너무 희귀하거나 너무 흔한 단어 제외
        max_features=5000,     # 최대 특성 수
        ngram_range=(1,2),     # 단어 및 2-그램 포함
        norm='l2',            # L2 정규화
        use_idf=True,         # IDF 사용
        smooth_idf=True        # IDF 스무딩 적용
    )
    tfidf_matrix = tfidf.fit_transform(documents)

    with open(vec_path, "wb") as f:
        pickle.dump(tfidf, f)
    with open(gv_path, "wb") as f:
        pickle.dump({gid: tfidf_matrix[i] for i, gid in enumerate(group_ids)}, f)
    print(f"✅ TF-IDF 학습·저장 완료 ({tfidf_matrix.shape})")
else:
    with open(vec_path, "rb") as f:
        tfidf = pickle.load(f)
    with open(gv_path, "rb") as f:
        group_vectors = pickle.load(f)
    tfidf_matrix = sparse.vstack([group_vectors[g] for g in group_vectors])
    group_ids = list(group_vectors.keys())
    print(f"🔄 TF-IDF 로드 완료 ({tfidf_matrix.shape})")

# group_vectors 딕트가 메모리에 없다면 다시 만들기
if "group_vectors" not in locals():
    group_vectors = {gid: tfidf_matrix[i] for i, gid in enumerate(group_ids)}

# ================================================================
# G. ALS 추천 함수 (안전 버전)
# ================================================================
def get_als_recommendations(user_index:int, N:int=100):
    """ALS 기반 상위 N개 asset_id 반환 (매핑 불일치 안전 처리)"""
    u_idx = user_map[user_index]
    item_ids, scores = als_model.recommend(
        u_idx, user_items[u_idx],
        N=N, filter_already_liked_items=True
    )
    rec_asset_ids, rec_scores = [], []
    for idx, sc in zip(item_ids, scores):
        aid = item_map_inv.get(idx)
        if aid is not None:
            rec_asset_ids.append(aid)
            rec_scores.append(sc)
    return rec_asset_ids, np.array(rec_scores)


🔄 형태소 분석 스킵 (이미 194379개 저장)
🔄 TF-IDF 로드 완료 ((194379, 5000))
🔄 TF-IDF 로드 완료 ((194379, 5000))


In [14]:
# ================================================================
# H. 하이브리드 추천 함수 (ALS + TF-IDF)
# ================================================================
from sklearn.metrics.pairwise import cosine_similarity

asset_to_group = content_df.set_index("asset_id")["group_id"].to_dict()    # ★추가
df["group_id"] = df["asset_id"].map(asset_to_group)  

# 시청 이력·그룹 이력
user_watched = df.groupby("user_index")["asset_id"].apply(list).to_dict()
user_groups  = df.groupby("user_index")["group_id"].apply(
    lambda x: list(set(x))
).to_dict()
asset_to_group = content_df.set_index("asset_id")["group_id"].to_dict()
asset_to_super = content_df.set_index("asset_id")["super_asset_nm"].to_dict()

# 사용자가 시청한 프로그램들(super_asset_nm) 목록 생성
user_watched_programs = df.groupby("user_index")["super_asset_nm"].apply(
    lambda x: set(x)
).to_dict()

# asset_id에 대한 poster_url 매핑 생성
asset_to_poster = df.set_index("asset_id")["poster_url"].to_dict()

In [15]:
# ================================================================
# J. 사용자 선호도 테이블 (genre / orgnl_cntry)
# ================================================================
user_genre_pref = (
    df.groupby(["user_index", "genre"]).size()
      .unstack(fill_value=0)
      .apply(lambda r: r / r.sum(), axis=1)
      .to_dict("index")
)

user_country_pref = (
    df.groupby(["user_index", "orgnl_cntry"]).size()
      .unstack(fill_value=0)
      .apply(lambda r: r / r.sum(), axis=1)
      .to_dict("index")
)

asset_to_genre   = df.set_index("asset_id")["genre"].to_dict()
asset_to_country = df.set_index("asset_id")["orgnl_cntry"].to_dict()

In [17]:
# ================================================================
# K. hybrid_recommend( ) – 가중치 통합 버전
# ================================================================
def hybrid_recommend(user_index:int, *,
                     N=100, M=100, K=10,
                     w_als=0.4, w_cont=0.3, w_genre=0.2, w_country=0.1):
    """ALS + TF-IDF + genre + country 선호도 가중 합산"""
    assert abs(w_als + w_cont + w_genre + w_country - 1) < 1e-6, \
        "가중치 합은 1이어야 합니다."

    watched        = set(user_watched.get(user_index, []))
    user_group_ids = user_groups.get(user_index, [])
    
    # 사용자가 시청한 프로그램 목록 가져오기
    watched_programs = user_watched_programs.get(user_index, set())

    # 1) ALS 후보
    als_assets, _  = get_als_recommendations(user_index, N=N)

    # 2) 콘텐츠 후보
    if user_group_ids:
        user_vecs = sparse.vstack([group_vectors[g] for g in user_group_ids])
        sims      = cosine_similarity(tfidf_matrix, user_vecs).max(axis=1)
        rank_idx  = np.argsort(sims)[::-1]
        rank_idx  = [i for i in rank_idx if group_ids[i] not in user_group_ids]
        top_groups = [group_ids[i] for i in rank_idx[:M]]
        content_assets = [rep_map[g]["asset_id"] for g in top_groups]
    else:
        sims, content_assets = np.zeros(len(group_ids)), []

    # 3) 후보 통합 (이미 시청한 asset_id는 제외)
    candidates = list(set(als_assets + content_assets) - watched)
    if not candidates:
        return []
    
    # 4) 사용자가 이미 시청한 프로그램은 후보에서 제외
    candidates = [aid for aid in candidates if asset_to_super.get(aid) not in watched_programs]
    if not candidates:
        return []

    # 5) ALS 점수 (0-1)
    u_idx   = user_map[user_index]
    uf      = als_model.user_factors[u_idx]
    cf_idx  = [item_map[a] for a in candidates]
    cf_mat  = als_model.item_factors[cf_idx]
    als_raw = (uf @ cf_mat.T).flatten()
    als_norm = (als_raw - als_raw.min()) / (als_raw.ptp() or 1)

    # 6) TF-IDF / genre / country 점수
    cont_scores   = np.array([
        sims[asset_to_group.get(a, -1)] if a in asset_to_group else 0
        for a in candidates
    ])
    genre_scores  = np.array([
        user_genre_pref.get(user_index, {}).get(
            asset_to_genre.get(a, None), 0.0)
        for a in candidates
    ])
    country_scores = np.array([
        user_country_pref.get(user_index, {}).get(
            asset_to_country.get(a, None), 0.0)
        for a in candidates
    ])

    # 7) 가중 합산
    hybrid = (w_als     * als_norm      +
              w_cont    * cont_scores   +
              w_genre   * genre_scores  +
              w_country * country_scores)
    
    # 8) 포스터 URL 유무로 구분하여 정렬
    candidates_with_scores = [(aid, hybrid[i], asset_to_poster.get(aid, None) is not None, asset_to_poster.get(aid, None)) 
                              for i, aid in enumerate(candidates)]
    
    # 포스터 있는 것은 True로 정렬 우선순위(내림차순), 그 다음 유사도 점수 내림차순
    candidates_with_scores.sort(key=lambda x: (not x[2], -x[1]))
    
    # 9) 상위 K개 (프로그램 중복 제거)
    seen_super, result = set(), []
    for aid, score, has_poster, poster_url in candidates_with_scores:
        sup = asset_to_super.get(aid)
        if sup and sup not in seen_super:
            result.append((aid, sup, score, poster_url))
            seen_super.add(sup)
            if len(result) == K: break
    return result

In [26]:
# ================================================================
# I. 사용 예시
# ================================================================
sample_user = df["user_index"].iloc[8]

# 사용자 정보 출력
print(f"사용자 ID: {sample_user}")

# 시청 이력 가져오기
watched_assets = user_watched.get(sample_user, [])
watched_progs = user_watched_programs.get(sample_user, set())

# 시청 이력이 너무 길면 잘라서 출력
max_display = 10  # 최대 출력 수
print(f"\n시청한 프로그램 ({len(watched_progs)} 개 중 {min(len(watched_progs), max_display)}개 표시):")
for i, prog in enumerate(list(watched_progs)[:max_display]):
    print(f"  {i+1}. {prog}")

# 하이브리드 추천 실행 - 같은 super_asset_nm을 갖는 VOD를 제외하고 추천
recs = hybrid_recommend(sample_user, w_als=0.25, w_cont=0.25, w_genre=0.3, w_country=0.2)

print("\n추천 결과 (포스터 이미지 있는 VOD 우선):")
for aid, sup, score, _ in recs:  # 포스터 URL은 출력하지 않음
    print(f"  asset_id = {aid:<8} | 프로그램명 = {sup:<20} | 유사도 = {score:.4f}")


사용자 ID: 8

시청한 프로그램 (26 개 중 10개 표시):
  1. 안싸우면 다행이야
  2. 꼬리에꼬리를무는그날이야기
  3. 살림하는 남자들 2
  4. 실화탐사대
  5. 동상이몽 2 너는 내 운명
  6. 낭만닥터 김사부2
  7. 여인천하
  8. 하늘의 인연
  9. 범죄도시2
  10. 전지적 참견 시점

추천 결과 (포스터 이미지 있는 VOD 우선):
  asset_id = M4660787LSGH46195501 | 프로그램명 = 하나뿐인 내편              | 유사도 = 0.6897
  asset_id = M4944203LSGK60619101 | 프로그램명 = 허준                   | 유사도 = 0.6380
  asset_id = M4349421LSGL16316501 | 프로그램명 = 푸른 바다의 전설            | 유사도 = 0.6206
  asset_id = M4946823LSGK29755901 | 프로그램명 = 상도                   | 유사도 = 0.6105
  asset_id = M4828614LSGH50030601 | 프로그램명 = 더 킹   영원의 군주         | 유사도 = 0.6072
  asset_id = M5089965LFOK51986701 | 프로그램명 = 태풍의 신부               | 유사도 = 0.6065
  asset_id = M5036846LFOJ21368301 | 프로그램명 = 불후의명곡2               | 유사도 = 0.6004
  asset_id = M4335743LSGK66120001 | 프로그램명 = 꽃보다할배 그리스            | 유사도 = 0.5987
  asset_id = M5031279LSGJ08502801 | 프로그램명 = 프리한19                | 유사도 = 0.5926
  asset_id = M4769363LSGL40630401 | 프로그램명 = VIP                  | 유사도